# CogVideoX-5B — Colab T4

**Универсальная генерация видео** с CogVideoX-5B (THUDM/ZhipuAI) на бесплатном T4 GPU.

### Почему CogVideoX?
| | CogVideoX-5B | Wan 2.2 14B | HunyuanVideo 1.5 |
|---|---|---|---|
| **Размер модели** | ~5 ГБ (INT8) | ~14 ГБ (FP8) | ~7.76 ГБ (FP8) |
| **Режимы** | T2V, I2V, Video extend | T2V, I2V, V2V, Talking | T2V |
| **Стиль** | Плавный, кинематографичный | Реалистичный | Реалистичный |
| **Скорость** | Быстрая | Медленная | Средняя |
| **Лучше всего для** | Стилизованный контент, разнообразие | Реализм, сложные сцены | Быстрые итерации |

CogVideoX создаёт **другую эстетику** — более плавную, кинематографичную. Отлично подходит для разнообразия в вашем видео-пайплайне.

**Запускайте ячейки 1-6 по порядку**, затем откройте ссылку на туннель.

In [ ]:
#@title 1. Проверка GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} ГБ")

In [ ]:
#@title 2. Установка ComfyUI + Кастомные ноды
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git 2>/dev/null || echo "Уже склонировано"
%cd /content/ComfyUI
!pip install -r requirements.txt -q

# Кастомные ноды
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/kijai/ComfyUI-CogVideoXWrapper.git 2>/dev/null || echo "Уже склонировано"
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git 2>/dev/null || echo "Уже склонировано"
!git clone https://github.com/kijai/ComfyUI-KJNodes.git 2>/dev/null || echo "Уже склонировано"

!pip install -r ComfyUI-CogVideoXWrapper/requirements.txt -q
!pip install -r ComfyUI-VideoHelperSuite/requirements.txt -q
!pip install -r ComfyUI-KJNodes/requirements.txt -q

print("\nУстановлено!")

In [ ]:
#@title 3. Скачивание моделей CogVideoX (~10 ГБ)
import os

M = "/content/ComfyUI/models"
os.makedirs(f"{M}/diffusion_models", exist_ok=True)
os.makedirs(f"{M}/text_encoders", exist_ok=True)
os.makedirs(f"{M}/vae", exist_ok=True)
os.makedirs(f"{M}/clip", exist_ok=True)

HF = "https://huggingface.co/Kijai/CogVideoX_comfy/resolve/main"

files = {
    # CogVideoX-5B transformer INT8 (~5 ГБ)
    f"{M}/diffusion_models/cogvideox_5b_I2V_fp8_e4m3fn.safetensors":
        f"{HF}/cogvideox_5b_I2V_fp8_e4m3fn.safetensors",
    # T5 XXL текстовый энкодер FP8 (~4.9 ГБ)
    f"{M}/text_encoders/t5xxl_fp8_e4m3fn.safetensors":
        "https://huggingface.co/mcmonkey/google_t5-v1_1-xxl_encoderonly/resolve/main/t5xxl_fp8_e4m3fn.safetensors",
    # CogVideoX VAE (~400 МБ)
    f"{M}/vae/cogvideox_vae.safetensors":
        f"{HF}/cogvideox_vae.safetensors",
}

for dest, url in files.items():
    if os.path.exists(dest):
        print(f"Уже существует: {os.path.basename(dest)}")
    else:
        print(f"\nСкачивание {os.path.basename(dest)}...")
        !wget -q --show-progress -O "{dest}" "{url}"

print("\nВсе модели готовы!")
!du -sh {M}/diffusion_models/* {M}/text_encoders/* {M}/vae/*

In [ ]:
#@title 4. Скачивание воркфлоу
import os

GIST = "ea1ab4a53e1be1b8401e5031bcdae4f3"
RAW = f"https://gist.githubusercontent.com/russiankendricklamar/{GIST}/raw"
WF_DIR = "/content/ComfyUI/user/default/workflows"
os.makedirs(WF_DIR, exist_ok=True)

!wget -q -O "{WF_DIR}/video_cogvideo_t2v.json" "{RAW}/video_cogvideo_t2v.json"
print("Воркфлоу скачан!")

In [ ]:
#@title 5. Загрузка медиа (фото для режима I2V)
from google.colab import files
import os

INPUT_DIR = "/content/ComfyUI/input"
os.makedirs(INPUT_DIR, exist_ok=True)

uploaded = files.upload()
for name, data in uploaded.items():
    path = os.path.join(INPUT_DIR, name)
    with open(path, "wb") as f:
        f.write(data)
    print(f"Сохранено: {path}")

print(f"\nФайлы в input/: {os.listdir(INPUT_DIR)}")

In [ ]:
#@title 6. Запуск ComfyUI
import subprocess, time, re

# Установка cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Запуск ComfyUI
comfy = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    cwd="/content/ComfyUI",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
print("Запуск ComfyUI... (30 сек)")
time.sleep(30)

# Cloudflare туннель
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8188"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
time.sleep(5)

url = None
for _ in range(20):
    line = tunnel.stdout.readline().decode()
    match = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        break

if url:
    print(f"\n{'='*60}")
    print(f"ComfyUI готов! Откройте: {url}")
    print(f"{'='*60}")
    print("\nЗагрузите воркфлоу: Меню -> Load -> video_cogvideo_t2v")
else:
    print("Ссылка на туннель не найдена. ComfyUI работает на порту 8188.")

In [ ]:
#@title 7. Сохранение на Google Drive
from google.colab import drive
import shutil, glob, os

drive.mount('/content/drive')

src = "/content/ComfyUI/output"
dst = "/content/drive/MyDrive/ComfyUI_Output/cogvideo"
os.makedirs(dst, exist_ok=True)

videos = glob.glob(f"{src}/*.mp4") + glob.glob(f"{src}/*.png")
if not videos:
    print("Результатов пока нет. Сначала сгенерируйте видео!")
else:
    for v in videos:
        shutil.copy2(v, dst)
        print(f"Скопировано: {os.path.basename(v)}")
    print(f"\n{len(videos)} файлов сохранено в {dst}")

---
## Руководство по использованию

### video_cogvideo_t2v — Текст-в-Видео
1. Откройте ссылку на туннель
2. Загрузите воркфлоу **video_cogvideo_t2v**
3. Отредактируйте текстовый промпт
4. Нажмите **Queue Prompt**

### Разрешение и длительность
CogVideoX-5B генерирует в разрешении **720x480** (или 480x720 портрет) по умолчанию.
- 49 кадров = ~6 сек при 8fps
- Оставьте 49 кадров для T4, чтобы избежать OOM

### Советы по промптам
CogVideoX хорошо реагирует на:
- **Описания сцен**: "A serene lake at sunset, mountains in the background, gentle ripples"
- **Экшн-сцены**: "A cat jumping from a shelf, slow motion, soft lighting"
- **Абстракции**: "Flowing liquid gold forming into geometric shapes, dark background"

### Режим I2V
Воркфлоу также поддерживает Image-to-Video:
1. Загрузите изображение в ячейке 5
2. В ComfyUI подключите ноду LoadImage к CogVideo sampler
3. Модель анимирует ваше изображение

### Сравнение с Wan 2.2
- CogVideoX: более плавное движение, кинематографичнее, ниже разрешение
- Wan 2.2: чётче детали, реалистичнее, выше разрешение
- Используйте обе модели для разнообразия контента!